# 10 - The CRM & Support MCP server directly: tools, resources, prompts and business rules

The agents reach customers and tickets only through the **FastMCP** server (streamable HTTP, SQLite, Docker). Here we
talk to it like any MCP client would - the same way Claude Desktop, VS Code or another agent framework could.

**Prerequisites:** `make mcp-server-up` (check with `make mcp-server-status`).

In [ ]:
# Setup: make src/ importable and load the settings from .env
import sys, warnings
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore", message=".*langchain.mcp.*")

from agentic_rag.config import AgentSettings

settings = AgentSettings.load(ROOT)
print(f"Coordinator: {settings.model} ({settings.coordinator_effort}) | specialists: {settings.specialist_model} "
      f"({settings.specialist_effort}) | judge: {settings.judge_model} (pass >= {settings.judge_pass_score})")

In [ ]:
import json
from fastmcp import Client

URL = settings.crm_mcp_url
async with Client(URL) as client:
    tools = await client.list_tools()
    resources = await client.list_resources()
    templates = await client.list_resource_templates()
    prompts = await client.list_prompts()
print("Tools:    ", [(t.name, "read-only" if t.annotations and t.annotations.read_only_hint else "write") for t in tools])
print("Resources:", [str(r.uri) for r in resources])
print("Templates:", [t.uri_template for t in templates])
print("Prompts:  ", [p.name for p in prompts])

## A support dashboard from resources

In [ ]:
async with Client(URL) as client:
    stats = json.loads((await client.read_resource("tickets://stats"))[0].text)
    customer = json.loads((await client.read_resource("crm://customers/CUST-0004"))[0].text)
    tickets = json.loads((await client.read_resource("crm://customers/CUST-0004/tickets"))[0].text)
print("Tickets by status  :", stats["by_status"])
print("Tickets by priority:", stats["by_priority"])
print("Top open models    :", stats.get("top_open_models"))
print(f"\n{customer['customer_id']} {customer['full_name']} active={customer['active_status']} - {customer['remarks']}")
for t in tickets["tickets"]:
    print(f"  {t['ticket_id']} [{t['status']}/{t['priority']}] {t['title']}")

## Case-insensitive partial search

In [ ]:
async with Client(URL) as client:
    people = await client.call_tool("search_customer_profiles", {"name": "SHETTY"})
    found = await client.call_tool("search_tickets", {"text": "charging", "status": "open", "limit": 5})
print([(c["customer_id"], c["full_name"], c["active_status"]) for c in people.structured_content["customers"]])
print(found.structured_content["total"], "open tickets mention 'charging'; first 5:")
for t in found.structured_content["tickets"]:
    print(f"  {t['ticket_id']} {t['product_model']}: {t['title']}")

## Prompts: server-side templates with live data

In [ ]:
async with Client(URL) as client:
    prompt = await client.get_prompt("triage_ticket", {"ticket_id": "TKT-000120"})
print(prompt.messages[0].content.text[:1200])

## Business rules are enforced by the server
Each call below is refused with a clear message; nothing is written.

In [ ]:
from fastmcp.exceptions import ToolError

attempts = [
    ("inactive customer", "register_ticket", {"customer_id": "CUST-0002", "title": "TV not turning on",
                                               "description": "No power", "priority": "high"}),
    ("invalid priority", "register_ticket", {"customer_id": "CUST-0001", "title": "Test", "description": "Test",
                                              "priority": "urgent"}),
    ("already resolved", "resolve_ticket", {"ticket_id": "TKT-000009", "resolved_by": "Demo",
                                             "resolution_remarks": "Demo"}),
    ("invalid status", "search_tickets", {"status": "pending"}),
]
async with Client(URL) as client:
    for label, tool, args in attempts:
        try:
            await client.call_tool(tool, args)
            print(f"{label:<18} -> accepted?!")
        except ToolError as exc:
            print(f"{label:<18} -> {exc}")

## The same tools as LangChain tools (what the agents get)

In [ ]:
from agentic_rag.mcp_client import load_crm_tools

crm = await load_crm_tools(settings)
print(crm.describe())

## What to observe
- Read tools carry `readOnlyHint`; write tools are annotated as writes - clients can ask for confirmation.
- Resources give a dashboard view without any tool call; prompts embed live records.
- Rules (inactive customers, enums, resolvable states) live in the server, so every client gets them.
- **No authentication** - the port is published on `127.0.0.1` only (see SECURITY.md).